In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/GP/CSV_completeDataset/

testing  training  valid


In [2]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

BOUNDARY = 0.5

def compute_bsa_bsre(y_true, out_scores, boundary=BOUNDARY):
    y_true = np.asarray(y_true).astype(int)
    out_scores = np.asarray(out_scores).astype(float)

    y_pred = (out_scores >= boundary).astype(int)

    Cs = (y_pred != y_true).astype(int)
    bsre = np.mean(Cs * np.abs(out_scores - boundary) ** 2)
    bsa = accuracy_score(y_true, y_pred)

    return bsa, bsre, y_pred


def compute_va_vre(y_true, out_scores, groups, boundary=BOUNDARY):
    y_true = np.asarray(y_true).astype(int)
    out_scores = np.asarray(out_scores).astype(float)
    groups = np.asarray(groups)

    video_true = []
    video_pred = []
    video_errors = []

    for vid in np.unique(groups):
        idx = np.where(groups == vid)[0]

        true_label = int(round(np.mean(y_true[idx])))
        avg_out = np.mean(out_scores[idx])
        pred_label = int(avg_out >= boundary)

        Cv = int(pred_label != true_label)

        video_true.append(true_label)
        video_pred.append(pred_label)
        video_errors.append(Cv * np.abs(avg_out - boundary) ** 2)

    va = accuracy_score(video_true, video_pred)
    vre = np.mean(video_errors)

    return va, vre, np.array(video_true), np.array(video_pred)

In [4]:
#this is the one is used with SHUFFLING(SHUFFLING now is commnted)
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.utils import shuffle
from scipy import stats
import random
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# after added it the accrucey become the same after retrain:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size, stride):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the -row block
                        window_features = features[i : i + window_size]

                        # Extract  corresponding labels
                        window_labels = row_labels[i : i + window_size]

                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        final_label = mode_result.mode[0]

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/CSV_completeDataset/training'
val_base   = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'
test_base = '/content/drive/MyDrive/GP/CSV_completeDataset/testing'


WINDOW_SIZE = 10    #10
STRIDE = 5      #2


X_train_raw, y_train = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_val_raw, y_val     = load_labeled_data(val_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

#X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=42)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")


# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_val_reshaped = X_val_raw.reshape(-1, 4)
X_val_scaled = scaler.transform(X_val_reshaped)
X_val = X_val_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

#use scaler.transform here, not fit_transform.

#because we want to scale your test data using the same average and rules learned from the training data.
# If use "fit" on the test data, it’s like "Data Leakage."

# =========================
# 5) 2clayer
# =========================
model = Sequential([
    # Input shape: (Time steps, Features)
    LSTM(96, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    LSTM(64),
    tf.keras.layers.Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') #Dense(1, activation='linear')

])

# =========================
# 6)
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    #loss='mse',
    #metrics=['mae']
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 7)
# =========================
model.fit(X_train, y_train,validation_data=(X_val, y_val), epochs=20, batch_size=32, verbose=1)
#save_path = '/content/drive/MyDrive/GP/LstmModels/best_drowsy_model.h5'
# 2. Save the entire model (Architecture + Weights + Optimizer state)
#model.save(save_path)

#print(f"🚀 Best model saved successfully to: {save_path}")

# =========================
# 8)
# =========================
# Initialize counters for the summary
total_windows_tested = 0
correct_windows = 0
total_videos_tested = 0
correct_videos = 0

all_video_labels = []
all_video_preds = []

#all_window_labels = []
#all_window_preds = []

global_window_scores = []
global_window_true = []
global_window_groups = []

print(f"{'File Name':<20} | {'Label':<8} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
print("-" * 70)


# We will loop through the testing folders to get a result for each file
for category in ['Drowsy', 'Alert']:
    folder_path = os.path.join(test_base, category)

    if not os.path.exists(folder_path):
        continue

    true_val = 1 if category == 'Drowsy' else 0

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            #  Load the individual file
            file_path = os.path.join(folder_path, filename)
            df_file = pd.read_csv(file_path)

            # Select features and check size
            feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
            if len(feats) < WINDOW_SIZE:
                continue

            # 3. Create windows for THIS file only
            file_windows = []
            file_labels = []
            for i in range(0, len(feats) - WINDOW_SIZE + 1, STRIDE):
                file_windows.append(feats[i : i + WINDOW_SIZE])
                file_labels.append(true_val)

            X_file = np.array(file_windows)

            # 4. Scale using the training scaler
            X_file_reshaped = X_file.reshape(-1, 4)
            X_file_scaled = scaler.transform(X_file_reshaped)
            X_file_final = X_file_scaled.reshape(-1, WINDOW_SIZE, 4)

            # 5. Model Prediction
            #file_preds = model.predict(X_file_final, verbose=0)
            #file_rounded = (file_preds > 0.5).astype(int).flatten()
            file_preds = model.predict(X_file_final, verbose=0).flatten()
            file_rounded = (file_preds > BOUNDARY).astype(int)

         # --- UPDATE WINDOW COUNTERS ---
            total_windows_tested += len(file_rounded)
            correct_windows += np.sum(file_rounded == true_val)

            global_window_scores.extend(file_preds)
            global_window_true.extend([true_val] * len(file_preds))
            global_window_groups.extend([filename] * len(file_preds))
            #all_window_preds.extend(file_rounded)
            #all_window_labels.extend(file_labels)

            # --- UPDATE VIDEO COUNTERS ---
            drowsy_percent = (np.sum(file_rounded) / len(file_rounded)) * 100
            verdict_val = 1 if drowsy_percent > 50 else 0

            total_videos_tested += 1
            if verdict_val == true_val:
                correct_videos += 1

            all_video_labels.append(true_val)
            all_video_preds.append(verdict_val)

            # Printing logic
            status = "✅" if verdict_val == true_val else "❌"
            v_text = "DROWSY" if verdict_val == 1 else "ALERT"
            print(f"{filename[:20]:<20} | {category:<8} | {drowsy_percent:>8.1f}% | {v_text:<8} | {status}")



# --- Calculate Accuracy Percentage ---

# Final Summary
win_acc = (correct_windows / total_windows_tested) * 100 if total_windows_tested > 0 else 0
vid_acc = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

print("\n" + "="*40)
print(f"WINDOW ACCURACY : {win_acc:.2f}%")
print(f"VIDEO ACCURACY:  {vid_acc:.2f}%")
print("="*40)
print(f"================================")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Training Windows: 739
Window Shape: 10 rows x 4 features


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.8065 - loss: 0.5582 - val_accuracy: 0.7481 - val_loss: 0.5681
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8457 - loss: 0.3272 - val_accuracy: 0.8296 - val_loss: 0.5100
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9039 - loss: 0.2348 - val_accuracy: 0.8444 - val_loss: 0.5577
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9107 - loss: 0.2204 - val_accuracy: 0.8519 - val_loss: 0.5439
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9175 - loss: 0.2043 - val_accuracy: 0.8444 - val_loss: 0.5991
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9256 - loss: 0.1867 - val_accuracy: 0.8370 - val_loss: 0.6440
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9256 - loss: 0.1836 - val_accuracy: 0.8370 - val_loss: 0.6397
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9283 - loss: 0.1751 - val_accuracy: 0.8370 - v

In [ ]:
bsa, bsre, y_window_pred = compute_bsa_bsre(global_window_true, global_window_scores)
print(f"Blink/Window Accuracy (BSA)       : {bsa * 100:.2f}%")
print(f"Blink/Window Squared Error (BSRE)  : {bsre:.5f}")
print("-" * 50)

# 2. Run your custom Video Level Metrics (VA, VRE) using the groups array
va, vre, video_true, video_pred = compute_va_vre(global_window_true, global_window_scores, global_window_groups)
print(f"Video Accuracy (VA)               : {va * 100:.2f}%")
print(f"Video Squared Error (VRE)         : {vre:.5f}")
print("="*50)

# 3. Print out clean text validation summaries for both tiers
print("\n📝 WINDOW-LEVEL CLASSIFICATION REPORT")
print(classification_report(global_window_true, y_window_pred, target_names=['Alert', 'Drowsy']))

print("\n📝 VIDEO-LEVEL CLASSIFICATION REPORT")
print(classification_report(video_true, video_pred, target_names=['Alert', 'Drowsy']))

Blink/Window Accuracy (BSA)       : 60.77%
Blink/Window Squared Error (BSRE)  : 0.06694
--------------------------------------------------
Video Accuracy (VA)               : 60.00%
Video Squared Error (VRE)         : 0.04098

📝 WINDOW-LEVEL CLASSIFICATION REPORT
              precision    recall  f1-score   support

       Alert       0.73      0.51      0.60       361
      Drowsy       0.53      0.74      0.62       266

    accuracy                           0.61       627
   macro avg       0.63      0.63      0.61       627
weighted avg       0.64      0.61      0.61       627


📝 VIDEO-LEVEL CLASSIFICATION REPORT
              precision    recall  f1-score   support

       Alert       0.67      0.40      0.50         5
      Drowsy       0.57      0.80      0.67         5

    accuracy                           0.60        10
   macro avg       0.62      0.60      0.58        10
weighted avg       0.62      0.60      0.58        10



In [ ]:
from sklearn.metrics import classification_report
print("\n" + "="*40)
print(f"WINDOW ACCURACY : {win_acc:.2f}%")
print(f"VIDEO ACCURACY:  {vid_acc:.2f}%")
print("="*40)
print("\n📊 WINDOW-LEVEL CLASSIFICATION REPORT")
print("-" * 40)
window_report = classification_report(all_window_labels, all_window_preds, target_names=['Alert', 'Drowsy'])
print(window_report)

print("\n🎬 VIDEO-LEVEL CLASSIFICATION REPORT")
print("-" * 40)
video_report = classification_report(all_video_labels, all_video_preds, target_names=['Alert', 'Drowsy'])
print(video_report)


WINDOW ACCURACY : 62.95%
VIDEO ACCURACY:  70.00%

📊 WINDOW-LEVEL CLASSIFICATION REPORT
----------------------------------------
              precision    recall  f1-score   support

       Alert       0.69      0.64      0.67       366
      Drowsy       0.56      0.61      0.58       271

    accuracy                           0.63       637
   macro avg       0.63      0.63      0.63       637
weighted avg       0.63      0.63      0.63       637


🎬 VIDEO-LEVEL CLASSIFICATION REPORT
----------------------------------------
              precision    recall  f1-score   support

       Alert       0.75      0.60      0.67         5
      Drowsy       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
report = classification_report(all_video_labels, all_video_preds,
                               target_names=['Alert', 'Drowsy'])

print(report)

              precision    recall  f1-score   support

       Alert       0.75      0.60      0.67         5
      Drowsy       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10



In [ ]:
#this is classfy the viduos without SHUFFLING
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.utils import shuffle
from scipy import stats
import random
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# after added it the accrucey become the same after retrain:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size=20, stride=5):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the -row block
                        window_features = features[i : i + window_size]

                        # Extract  corresponding labels
                        window_labels = row_labels[i : i + window_size]

                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        final_label = mode_result.mode[0]

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/CSV_completeDataset/training'
val_base   = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'
test_base  = '/content/drive/MyDrive/GP/CSV_completeDataset/testing'

WINDOW_SIZE = 10
STRIDE = 2


X_train_raw, y_train = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)

X_val_raw, y_val = load_labeled_data(val_base, WINDOW_SIZE, STRIDE)

X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")


# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_val_reshaped = X_val_raw.reshape(-1, 4)
X_val_scaled = scaler.transform(X_val_reshaped)
X_val = X_val_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

#use scaler.transform here, not fit_transform.

#because we want to scale your test data using the same average and rules learned from the training data.
# If use "fit" on the test data, it’s like "Data Leakage."

# =========================
# 5) 2clayer
# =========================
model = Sequential([
    # Input shape: (Time steps, Features)
    LSTM(32, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    LSTM(16),
    tf.keras.layers.Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') #Dense(1, activation='linear')

])

# =========================
# 6)
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    #loss='mse',
    #metrics=['mae']
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 7)
# =========================
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

# =========================
# 8)
# =========================
# Initialize counters for the summary
total_windows_tested = 0
correct_windows = 0
total_videos_tested = 0
correct_videos = 0
all_video_labels = []
all_video_preds = []


print(f"{'File Name':<20} | {'Label':<8} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
print("-" * 70)


# We will loop through the testing folders to get a result for each file
for category in ['Drowsy', 'Alert']:
    folder_path = os.path.join(test_base, category)

    if not os.path.exists(folder_path):
        continue

    true_val = 1 if category == 'Drowsy' else 0

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            #  Load the individual file
            file_path = os.path.join(folder_path, filename)
            df_file = pd.read_csv(file_path)

            # Select features and check size
            feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
            if len(feats) < WINDOW_SIZE:
                continue

            # 3. Create windows for THIS file only
            file_windows = []
            for i in range(0, len(feats) - WINDOW_SIZE + 1, STRIDE):
                file_windows.append(feats[i : i + WINDOW_SIZE])

            X_file = np.array(file_windows)

            # 4. Scale using the training scaler
            X_file_reshaped = X_file.reshape(-1, 4)
            X_file_scaled = scaler.transform(X_file_reshaped)
            X_file_final = X_file_scaled.reshape(-1, WINDOW_SIZE, 4)

            # 5. Model Prediction
            file_preds = model.predict(X_file_final, verbose=0)
            file_rounded = (file_preds > 0.5).astype(int).flatten()

         # --- UPDATE WINDOW COUNTERS ---
            total_windows_tested += len(file_rounded)
            correct_windows += np.sum(file_rounded == true_val)

            # --- UPDATE VIDEO COUNTERS ---
            drowsy_percent = (np.sum(file_preds) / len(file_preds)) * 100
            verdict_val = 1 if drowsy_percent > 50 else 0

            total_videos_tested += 1
            if verdict_val == true_val:
                correct_videos += 1

            all_video_labels.append(true_val)
            all_video_preds.append(verdict_val)

            # Printing logic
            status = "✅" if verdict_val == true_val else "❌"
            v_text = "DROWSY" if verdict_val == 1 else "ALERT"
            print(f"{filename[:20]:<20} | {category:<8} | {drowsy_percent:>8.1f}% | {v_text:<8} | {status}")



# --- Calculate Accuracy Percentage ---

# Final Summary
win_acc = (correct_windows / total_windows_tested) * 100 if total_windows_tested > 0 else 0
vid_acc = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

print("\n" + "="*40)
print(f"WINDOW ACCURACY : {win_acc:.2f}%")
print(f"VIDEO ACCURACY:  {vid_acc:.2f}%")
print("="*40)
print(f"================================")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Training Windows: 1805
Window Shape: 10 rows x 4 features
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.4510 - loss: 0.6962 - val_accuracy: 0.6422 - val_loss: 0.6916
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6349 - loss: 0.6875 - val_accuracy: 0.7309 - val_loss: 0.6849
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7285 - loss: 0.6759 - val_accuracy: 0.7064 - val_loss: 0.6781
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7601 - loss: 0.6614 - val_accuracy: 0.7003 - val_loss: 0.6672
Epoch 5/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8078 - loss: 0.6371 - val_accuracy: 0.7217 - val_loss: 0.6476
Epoch 6/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8260 - loss: 0.6049 - val_accuracy: 0.7339 - val_loss: 0.6186
Epoch 7/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8366 - loss: 0.5566 - val_accuracy: 0.7462 - val_loss: 0.5774
Epoch 8/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8576 - loss: 0.5004 - val_accuracy: 0.7584 - val_loss: 0.5

**hyperparameter search**

In [ ]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.4 MB/s eta 0:00:00


In [ ]:
#clear the fuolder
import shutil
if os.path.exists('tuning_logs'):
    shutil.rmtree('tuning_logs')

In [ ]:
#hyperparameter search:
#this is the one is used with SHUFFLING
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from scipy import stats
import random
import json
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# after added it the accrucey become the same after retrain:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size, stride):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the -row block
                        window_features = features[i : i + window_size]

                        # Extract  corresponding labels
                        window_labels = row_labels[i : i + window_size]
                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        #final_label = mode_result.mode[0]
                        final_label= np.bincount(window_labels).argmax()

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/CSV_completeDataset/training'
val_base = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'
test_base = '/content/drive/MyDrive/GP/CSV_completeDataset/testing'


WINDOW_SIZE = 10
STRIDE = 5


X_train_raw, y_train_raw = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_val_raw, y_val = load_labeled_data(val_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

#X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=42)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Total Training lable: { y_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")

# DIVIDE Training into Train (80%) and Validation (20%)
#X_train_split, X_val_split, y_train, y_val = train_test_split(
    #X_train_raw, y_train_raw, test_size=0.20, random_state=42
#)

# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_val_reshaped = X_val_raw.reshape(-1, 4)
X_val_scaled = scaler.transform(X_val_reshaped)
X_val = X_val_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

#use scaler.transform here, not fit_transform.

#because we want to scale your test data using the same average and rules learned from the training data.
# If use "fit" on the test data, it’s like "Data Leakage."

# =========================
def build_model(hp):
    model = Sequential()
    hp.Choice('batch_size', values=[16, 32, 64])
    # Tune LSTM Layers
    model.add(LSTM(units=hp.Int('units_1', 32, 128, step=32),
                   return_sequences=True,
                   input_shape=(WINDOW_SIZE, 4)))

    model.add(LSTM(units=hp.Int('units_2', 16, 64, step=16)))

    # Tune Regularization
    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))

    # Tune Dense Layer
    model.add(Dense(hp.Int('dense_units', 16, 64, step=16), activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    # Tune Learning Rate
    lr = hp.Choice('learning_rate', values=[1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

class MyBayesianTuner(kt.BayesianOptimization):
    def run_trial(self, trial, *args, **kwargs):
        # Dynamically fetch the choice picked by the tuner and apply it to fit()
        kwargs['batch_size'] = trial.hyperparameters.Choice('batch_size', values=[16, 32, 64])
        return super(MyBayesianTuner, self).run_trial(trial, *args, **kwargs)

# 6) Initialize Bayesian Optimization Tuner
#tuner = kt.BayesianOptimization(
tuner = MyBayesianTuner(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='tuning_logs',
    project_name='drowsy_driver_optimization_final'
)

# 7) Execute Tuning Search
tuner.search(X_train, y_train_raw, epochs=50, validation_data=(X_val, y_val), verbose=1)

# --- SAVE ALL HYPERPARAMETER TRIALS TO CSV ---
all_trials_data = []
# Retrieve all trials from the search
for trial_id in tuner.oracle.trials:
    trial = tuner.oracle.trials[trial_id]
    trial_info = trial.hyperparameters.values
    trial_info['val_accuracy'] = trial.score
    trial_info['trial_id'] = trial_id
    all_trials_data.append(trial_info)

results_df = pd.DataFrame(all_trials_data)
results_df.to_csv('/content/drive/MyDrive/GP/LstmModels/Lstm1_all_hyperparameter(with batch+new data)_trials.csv', index=False)
print("✅ All trials saved to 'all_hyperparameter_trials.csv' on Drive.")

# Get the absolute best model
model = tuner.get_best_models(num_models=1)[0]

# =========================
# 8)
# =========================
# Initialize counters for the summary
total_windows_tested = 0
correct_windows = 0
total_videos_tested = 0
correct_videos = 0
all_video_labels = []
all_video_preds = []
# Added global arrays to feed into custom formulas
global_window_true = []
global_window_scores = []
global_window_groups = []


print(f"{'File Name':<20} | {'Label':<8} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
print("-" * 70)


# We will loop through the testing folders to get a result for each file
for category in ['Drowsy', 'Alert']:
    folder_path = os.path.join(test_base, category)

    if not os.path.exists(folder_path):
        continue

    true_val = 1 if category == 'Drowsy' else 0

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            #  Load the individual file
            file_path = os.path.join(folder_path, filename)
            df_file = pd.read_csv(file_path)

            # Select features and check size
            feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
            if len(feats) < WINDOW_SIZE:
                continue

            # 3. Create windows for THIS file only
            file_windows = []
            for i in range(0, len(feats) - WINDOW_SIZE + 1, STRIDE):
                file_windows.append(feats[i : i + WINDOW_SIZE])

            X_file = np.array(file_windows)

            # 4. Scale using the training scaler
            X_file_reshaped = X_file.reshape(-1, 4)
            X_file_scaled = scaler.transform(X_file_reshaped)
            X_file_final = X_file_scaled.reshape(-1, WINDOW_SIZE, 4)

            # 5. Model Prediction
            #file_preds = model.predict(X_file_final, verbose=0)
            #file_rounded = (file_preds > 0.5).astype(int).flatten()
            file_preds = model.predict(X_file_final, verbose=0).flatten()
            file_rounded = (file_preds >= 0.5).astype(int)


         # --- UPDATE WINDOW COUNTERS ---
            total_windows_tested += len(file_rounded)
            correct_windows += np.sum(file_rounded == true_val)

            # --- UPDATE VIDEO COUNTERS ---
            drowsy_percent = (np.sum(file_rounded) / len(file_rounded)) * 100
            verdict_val = 1 if drowsy_percent > 50 else 0

            total_videos_tested += 1
            if verdict_val == true_val:
                correct_videos += 1

            all_video_labels.append(true_val)
            all_video_preds.append(verdict_val)

            # Printing logic
            status = "✅" if verdict_val == true_val else "❌"
            v_text = "DROWSY" if verdict_val == 1 else "ALERT"
            print(f"{filename[:20]:<20} | {category:<8} | {drowsy_percent:>8.1f}% | {v_text:<8} | {status}")



# --- Calculate Accuracy Percentage ---

# Final Summary
win_acc = (correct_windows / total_windows_tested) * 100 if total_windows_tested > 0 else 0
vid_acc = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

print("\n" + "="*40)
print(f"WINDOW ACCURACY : {win_acc:.2f}%")
print(f"VIDEO ACCURACY:  {vid_acc:.2f}%")
print("="*40)
print(f"================================")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Training Windows: 739
Total Training lable: 739
Window Shape: 10 rows x 4 features
Reloading Tuner from tuning_logs/drowsy_driver_optimization_final/tuner0.json
✅ All trials saved to 'all_hyperparameter_trials.csv' on Drive.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


File Name            | Label    | Drowsy %   | Verdict  | Status
----------------------------------------------------------------------
D022_20260513_190626 | Drowsy   |    100.0% | DROWSY   | ✅
D023_20260513_195650 | Drowsy   |    100.0% | DROWSY   | ✅
D024_20260513_212814 | Drowsy   |     78.8% | DROWSY   | ✅
D025_20260513_233342 | Drowsy   |     30.4% | ALERT    | ❌
D026_20260514_025307 | Drowsy   |     70.8% | DROWSY   | ✅
A022_20260513_190541 | Alert    |     91.3% | DROWSY   | ❌
A023_20260513_195602 | Alert    |     16.7% | ALERT    | ✅
A024_20260513_212736 | Alert    |     95.5% | DROWSY   | ❌
A025_20260513_233235 | Alert    |     15.8% | ALERT    | ✅
A026_20260514_025227 | Alert    |     12.5% | ALERT    | ✅

WINDOW ACCURACY : 65.48%
VIDEO ACCURACY:  70.00%


In [ ]:
tuner.results_summary()

Results summary
Results in tuning_logs/drowsy_driver_optimization_final
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 07 summary
Hyperparameters:
batch_size: 32
units_1: 96
units_2: 64
dropout: 0.2
dense_units: 32
learning_rate: 0.001
val_accuracy: 0.8308605551719666
trial_id: 07
Score: 0.8308605551719666

Trial 02 summary
Hyperparameters:
batch_size: 32
units_1: 32
units_2: 16
dropout: 0.4
dense_units: 48
learning_rate: 0.001
val_accuracy: 0.8278931975364685
trial_id: 02
Score: 0.8278931975364685

Trial 01 summary
Hyperparameters:
batch_size: 32
units_1: 64
units_2: 64
dropout: 0.30000000000000004
dense_units: 64
learning_rate: 0.001
val_accuracy: 0.8278931975364685
trial_id: 01
Score: 0.8278931975364685

Trial 04 summary
Hyperparameters:
batch_size: 16
units_1: 96
units_2: 32
dropout: 0.2
dense_units: 16
learning_rate: 0.001
val_accuracy: 0.8249258399009705
trial_id: 04
Score: 0.8249258399009705

Trial 03 summary
Hyperparameters:
batch_size: 32
units_1

In [ ]:
# 1) Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# 2) Print them in a clean format
print("="*30)
print("🏆 BEST HYPERPARAMETERS")
print("="*30)
print(f"LSTM Units 1:    {best_hps.get('units_1')}")
print(f"LSTM Units 2:    {best_hps.get('units_2')}")
print(f"Dropout Rate:    {best_hps.get('dropout'):.2f}")
print(f"Dense Units:     {best_hps.get('dense_units')}")
print(f"Learning Rate:   {best_hps.get('learning_rate')}")
print("="*30)

# 3) Optionally, check the best score reached
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
print(f"Best Validation Accuracy: {best_trial.score * 100:.2f}%")
print("="*30)
save_path = '/content/drive/MyDrive/GP/LstmModels/best_tuned_lstm_model_final.h5'

# 3) Save it
model.save(save_path)

print(f"✅ Best model successfully saved to: {save_path}")

🏆 BEST HYPERPARAMETERS
LSTM Units 1:    96
LSTM Units 2:    64
Dropout Rate:    0.20
Dense Units:     32
Learning Rate:   0.001
Best Validation Accuracy: 83.09%
✅ Best model successfully saved to: /content/drive/MyDrive/GP/LstmModels/best_tuned_lstm_model_final.h5


optuna tuning

In [ ]:
!pip install optuna

In [ ]:
#hyperparameter search:
#this is the one is used with SHUFFLING
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import optuna
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from scipy import stats
import random
import json
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# after added it the accrucey become the same after retrain:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size, stride):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the -row block
                        window_features = features[i : i + window_size]

                        # Extract  corresponding labels
                        window_labels = row_labels[i : i + window_size]
                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        #final_label = mode_result.mode[0]
                        final_label= np.bincount(window_labels).argmax()

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/CSV_completeDataset/training'
val_base = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'
test_base = '/content/drive/MyDrive/GP/CSV_completeDataset/testing'


WINDOW_SIZE = 10
STRIDE = 5


X_train_raw, y_train_raw = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_val_raw, y_val = load_labeled_data(val_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

#X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=42)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Total Training lable: { y_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")

# DIVIDE Training into Train (80%) and Validation (20%)
#X_train_split, X_val_split, y_train, y_val = train_test_split(
    #X_train_raw, y_train_raw, test_size=0.20, random_state=42
#)

# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_val_reshaped = X_val_raw.reshape(-1, 4)
X_val_scaled = scaler.transform(X_val_reshaped)
X_val = X_val_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

# OPTUNA CONFIGURATION (Replaces Keras Tuner / MyBayesianTuner)
# ==============================================================================
class KerasPruningCallback(tf.keras.callbacks.Callback):
    def __init__(self, trial, monitor='val_accuracy'):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val = logs.get(self.monitor)
        if current_val is None:
            return

        # Report metric snapshot to the Optuna Engine
        self.trial.report(current_val, epoch)

        # Prune unpromising trial paths dynamically
        if self.trial.should_prune():
            raise optuna.TrialPruned(f"Trial pruned at epoch {epoch}.")


def objective(trial):
    # 1. Suggest hyperparameters using Optuna
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    units_1 = trial.suggest_int('units_1', 32, 128, step=32)
    units_2 = trial.suggest_int('units_2', 16, 64, step=16)
    dropout = trial.suggest_float('dropout', 0.2, 0.5, step=0.1)
    dense_units = trial.suggest_int('dense_units', 16, 64, step=16)
    lr = trial.suggest_categorical('learning_rate', [1e-3, 1e-4, 5e-5])

    # 2. Re-create the network architecture per trial
    model = Sequential()
    model.add(LSTM(units=units_1, return_sequences=True, input_shape=(WINDOW_SIZE, 4)))
    model.add(LSTM(units=units_2))
    model.add(Dropout(dropout))
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Optuna Pruning callback safely handles early stopping if trials show poor results
    pruning_callback = KerasPruningCallback(trial, 'val_accuracy')

    # 3. Train the model
    history = model.fit(
        X_train, y_train_raw,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=[pruning_callback],
        verbose=0  # Cleans up the console outputs during long tuning sessions
    )

    # Find the best val_accuracy reached during this model's lifetime
    val_accuracy = max(history.history['val_accuracy'])
    return val_accuracy


# Initialize the study optimization objective direction
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())

# Execute Tuning Search (equivalent to max_trials=10)
study.optimize(objective, n_trials=10)

# --- SAVE ALL HYPERPARAMETER TRIALS TO CSV ---
results_df = study.trials_dataframe()
# Renaming columns to match your prior script expectations downstream
results_df = results_df.rename(columns={'value': 'val_accuracy', 'number': 'trial_id'})
# Stripping Optuna prefixes 'params_' for clean visibility
results_df.columns = results_df.columns.str.replace('params_', '')

csv_export_path = '/content/drive/MyDrive/GP/LstmModels/Lstm1_all_hyperparameter(optuna)_trials.csv'
results_df.to_csv(csv_export_path, index=False)
print(f"✅ All trials saved to {csv_export_path} on Drive.")


# --- RETRAIN OR EXTRACT THE ABSOLUTE BEST MODEL ---
best_params = study.best_params
print(f"Best Hyperparameters: {best_params}")

# Build the definitive best model using found params
model = Sequential()
model.add(LSTM(units=best_params['units_1'], return_sequences=True, input_shape=(WINDOW_SIZE, 4)))
model.add(LSTM(units=best_params['units_2']))
model.add(Dropout(best_params['dropout']))
model.add(Dense(best_params['dense_units'], activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Fit one definitive run to handle evaluation predictions
model.fit(X_train, y_train_raw, epochs=50, batch_size=best_params['batch_size'], validation_data=(X_val, y_val), verbose=1)

model_save_path = '/content/drive/MyDrive/GP/LstmModels/Lstm_lamya/best_lstm_drowsy_model.keras'
model.save(model_save_path)
print(f"✅ Best model successfully saved to Google Drive at: {model_save_path}")
# ==============================================================================
# 8) Testing Evaluation Metrics Loops (Unchanged)
# ==============================================================================
total_windows_tested = 0
correct_windows = 0
total_videos_tested = 0
correct_videos = 0
all_video_labels = []
all_video_preds = []

print(f"{'File Name':<20} | {'Label':<8} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
print("-" * 70)

for category in ['Drowsy', 'Alert']:
    folder_path = os.path.join(test_base, category)
    if not os.path.exists(folder_path):
        continue

    true_val = 1 if category == 'Drowsy' else 0

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            df_file = pd.read_csv(file_path)

            feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
            if len(feats) < WINDOW_SIZE:
                continue

            file_windows = []
            for i in range(0, len(feats) - WINDOW_SIZE + 1, STRIDE):
                file_windows.append(feats[i : i + WINDOW_SIZE])

            X_file = np.array(file_windows)

            X_file_reshaped = X_file.reshape(-1, 4)
            X_file_scaled = scaler.transform(X_file_reshaped)
            X_file_final = X_file_scaled.reshape(-1, WINDOW_SIZE, 4)

            file_preds = model.predict(X_file_final, verbose=0)
            file_rounded = (file_preds > 0.5).astype(int).flatten()

            total_windows_tested += len(file_rounded)
            correct_windows += np.sum(file_rounded == true_val)

            drowsy_percent = (np.sum(file_rounded) / len(file_rounded)) * 100
            verdict_val = 1 if drowsy_percent > 50 else 0

            total_videos_tested += 1
            if verdict_val == true_val:
                correct_videos += 1

            all_video_labels.append(true_val)
            all_video_preds.append(verdict_val)

            status = "✅" if verdict_val == true_val else "❌"
            v_text = "DROWSY" if verdict_val == 1 else "ALERT"
            print(f"{filename[:20]:<20} | {category:<8} | {drowsy_percent:>8.1f}% | {v_text:<8} | {status}")

win_acc = (correct_windows / total_windows_tested) * 100 if total_windows_tested > 0 else 0
vid_acc = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

print("\n" + "="*40)
print(f"WINDOW ACCURACY : {win_acc:.2f}%")
print(f"VIDEO ACCURACY:  {vid_acc:.2f}%")
print("="*40)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[I 2026-05-24 13:02:14,518] A new study created in memory with name: no-name-f4072bb4-d47b-4048-9c8d-ed98155b7651


Total Training Windows: 739
Total Training lable: 739
Window Shape: 10 rows x 4 features


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-05-24 13:02:57,240] Trial 0 finished with value: 0.8518518805503845 and parameters: {'batch_size': 16, 'units_1': 64, 'units_2': 48, 'dropout': 0.2, 'dense_units': 16, 'learning_rate': 0.0001}. Best is trial 0 with value: 0.8518518805503845.
[I 2026-05-24 13:03:17,831] Trial 1 finished with value: 0.8518518805503845 and parameters: {'batch_size': 32, 'units_1': 64, 'units_2': 16, 'dropout': 0.5, 'dense_units': 48, 'learning_rate': 0.0001}. Best is trial 0 with value: 0.8518518805503845.
[I 2026-05-24 13:03:37,574] Trial 2 finished with value: 0.8444444537162781 and parameters: {'batch_size': 32, 'units_1': 32, 'units_2': 48, 'dropout': 0.5, 'dense_units': 16, 'learning_rate': 0.0001}. Best is tria

✅ All trials saved to /content/drive/MyDrive/GP/LstmModels/Lstm1_all_hyperparameter(optuna)_trials.csv on Drive.
Best Hyperparameters: {'batch_size': 32, 'units_1': 32, 'units_2': 48, 'dropout': 0.5, 'dense_units': 32, 'learning_rate': 0.001}
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.7037 - loss: 0.6350 - val_accuracy: 0.7407 - val_loss: 0.5688
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7970 - loss: 0.4554 - val_accuracy: 0.7556 - val_loss: 0.5483
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8525 - loss: 0.3423 - val_accuracy: 0.7852 - val_loss: 0.5066
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8904 - loss: 0.2698 - val_accuracy: 0.8222 - val_loss: 0.5126
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9107 - loss: 0.2244 - val_accuracy: 0.8296 - val_loss: 0.5886
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9120 - loss: 0.2103 - val_accuracy: 0.8444 - val_loss: 0.5908
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9161 - loss: 0.2100 - val_accuracy: 0.8519 - val_loss: 0.5547
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9175 - loss: 0.2032 - val_accuracy: 0.8519 - val_loss: 0.

**--------------------------------------------------------------------------------------------------------------------------**

In [ ]:
#this only classfy the windows
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.utils import shuffle
from scipy import stats
import random
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# after added it the accrucey become the same after retrain:
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size=20, stride=5):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the 20-row block
                        window_features = features[i : i + window_size]

                        # Extract the 20 corresponding labels
                        window_labels = row_labels[i : i + window_size]

                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        final_label = mode_result.mode[0]

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/upated_CSV/training'
test_base = '/content/drive/MyDrive/GP/upated_CSV/testing'

                     # with  seed:
WINDOW_SIZE = 10     #i tested size=8, stide=2 but the accurcy = 65.54%
STRIDE = 2           #i tested size=20, stide=5 but the accurcy = 65.73
                     #i tested size=10, stide=2 but the accurcy = 68.33%

                     # without  seed:
                     #size=8, stide=2, accurcy = 66.10%-->
                      #size=20, stide=5, accurcy = 63.50% --> 71.48%
                     # size=10, stide=2, accurcy =  68.33%-->

X_train_raw, y_train = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")


# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

#use scaler.transform here, not fit_transform.

#because we want to scale your test data using the same average and rules learned from the training data.
# If use "fit" on the test data, it’s like "Data Leakage."

# =========================
# 5) 2clayer
# =========================
model = Sequential([
    # Input shape: (Time steps, Features)
    LSTM(32, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    LSTM(16),
    tf.keras.layers.Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') #Dense(1, activation='linear')

])

# =========================
# 6)
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    #loss='mse',
    #metrics=['mae']
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 7)
# =========================
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# =========================
# 8)
# =========================
pred = model.predict(X_test)


# --- Calculate Accuracy Percentage ---

rounded_preds = (pred > 0.5).astype(int).flatten()

# 2. Count how many matches we have
#correct_predictions = np.sum(rounded_preds.flatten() == y_test)
correct_predictions = np.sum(rounded_preds.ravel() == y_test.ravel())
total_predictions = len(y_test)

# 3. Calculate percentage
accuracy_percentage = (correct_predictions / total_predictions) * 100

# Display results
print("\n--- Sample Results (Actual vs Predicted) ---")
for i in range(min(15, len(y_test))):
    print(f"Window {i+1} | Actual Label: {y_test[i]} | Predicted: {rounded_preds[i]}")

# Display the LAST 15 results (The Alert class)
print("\n--- Alert Results (Actual vs Predicted) ---")
total_test_samples = len(y_test)
start_index = max(0, total_test_samples - 15)
for i in range(start_index, total_test_samples):
    if i >= 0: # Safety check
        print(f"Window {i+1} | Actual Label: {y_test[i]} | Predicted: {rounded_preds[i]}")

print(f"\n================================")
print(f"Final Accuracy: {accuracy_percentage:.2f}%")
print(f"================================")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Training Windows: 1134
Window Shape: 10 rows x 4 features
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4268 - loss: 0.6995
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5300 - loss: 0.6914
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5705 - loss: 0.6844
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6155 - loss: 0.6784
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6252 - loss: 0.6706
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6384 - loss: 0.6631
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6402 - loss: 0.6490
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6658 - loss: 0.6376
Epoch 9/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6869 - loss: 0.6190
Epoch 10/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7002 - loss: 0.6004
Epoch 11/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7213 - loss: 0.5829
Epoch 12/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7275 - loss: 

**------------break----------------------------**

In [ ]:
#this is the one is used with SHUFFLING
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.utils import shuffle
from scipy import stats
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path, window_size=20, stride=5):
    #all_features = []
    #all_labels = []
    X_windows = [] #stor the feauters
    y_windows = [] #stor the labels

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 1, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
              try:
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                #all_features.append(features)

                # Assign the folder's label to every row in this file
                row_labels = np.full((features.shape[0],), label_value)
                #all_labels.append(labels)

                if len(features) < window_size:
                        continue

                    # 3. Create Windows
                for i in range(0, len(features) - window_size + 1, stride):
                        # Extract the 20-row block
                        window_features = features[i : i + window_size]

                        # Extract the 20 corresponding labels
                        window_labels = row_labels[i : i + window_size]

                        # Take the Most Frequent label for this window
                        #  using stats.mode to handles the "majority vote" logic
                        mode_result = stats.mode(window_labels, keepdims=True)
                        final_label = mode_result.mode[0]

                        X_windows.append(window_features)
                        y_windows.append(final_label)

              except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows)

    #return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/csv_files/training'
test_base = '/content/drive/MyDrive/GP/csv_files/testing'

WINDOW_SIZE = 20
STRIDE = 5 # test diffrent valeus

X_train_raw, y_train = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

# --- SHUFFLING STEP ---
# This mixes Drowsy and Alert samples so the model learns both at the same time
X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=42)
X_test_raw, y_test = shuffle(X_test_raw, y_test, random_state=42)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No windows were created!")
    print(f"Current shape is {X_train_raw.shape}. Possible reasons:")
    print(f"1. Your CSV files have fewer than {WINDOW_SIZE} rows.")
    print("2. The file paths are incorrect.")
else:
    print(f"Total Training Windows: {X_train_raw.shape[0]}")
    print(f"Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")


# 4) Preprocessing & Reshaping
scaler = StandardScaler()

# Flatten to 2D to fit the scaler, then reshape back to 3D
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

#use scaler.transform here, not fit_transform.

#because we want to scale your test data using the same average and rules learned from the training data.
# If use "fit" on the test data, it’s like "Data Leakage."

# =========================
# 5) 2clayer
# =========================
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# =========================
# 6)
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    #loss='mse',
    #metrics=['mae']
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 7)
# =========================
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# =========================
# 8)
# =========================
pred = model.predict(X_test)

# Display results
print("\n--- Sample Results (Actual vs Predicted) ---")
for i in range(min(15, len(y_test))):
    print(f"Window {i+1} | Actual Label: {y_test[i]:<5} | Predicted: {pred[i][0]:.4f}")
# Display the LAST 15 results (The Alert class)
print("\n--- Alert Results (Actual vs Predicted) ---")
total_test_samples = len(y_test)
for i in range(total_test_samples - 15, total_test_samples):
    if i >= 0: # Safety check
        print(f"Window {i+1} | Actual Label: {y_test[i]:<5} | Predicted: {pred[i][0]:.4f}")

# --- Calculate Accuracy Percentage ---
rounded_preds = (pred > 0.5).astype(int).flatten()

# 2. Count how many matches we have
correct_predictions = np.sum(rounded_preds.flatten() == y_test)
total_predictions = len(y_test)

# 3. Calculate percentage
accuracy_percentage = (correct_predictions / total_predictions) * 100

print(f"\n================================")
print(f"Final Accuracy: {accuracy_percentage:.2f}%")
print(f"================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Training Windows: 2606
Window Shape: 20 rows x 4 features
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


82/82 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7748 - loss: 0.6445
Epoch 2/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8231 - loss: 0.4518
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8450 - loss: 0.3963
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8622 - loss: 0.3644
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8676 - loss: 0.3526
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8684 - loss: 0.3483
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8695 - loss: 0.3421
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8738 - loss: 0.3365
Epoch 9/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8738 - loss: 0.3345
Epoch 10/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8738 - loss: 0.3311
Epoch 11/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8761 - loss: 0.3276
Epoch 12/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8734 - loss: 0.3319


**-------------------ignore the rest--------------------**

In [ ]:
#old
# =========================
# 1)
# =========================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from google.colab import drive

# =========================

# 1) Mount Drive
drive.mount('/content/drive')

# 2) Function to load data and assign labels based on folder
def load_labeled_data(base_path):
    all_features = []
    all_labels = []

    # Define your mapping
    # Folder "Drowsy" -> Label 1
    # Folder "Alert"  -> Label 0
    #categories = {'Drowsy': 1, 'Alert': 0}
    categories = {'Drowsy': 10, 'Alert': 0}

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} not found.")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
                df = pd.read_csv(os.path.join(folder_path, filename))
                # Selecting your specific columns
                features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                all_features.append(features)

                # Assign the folder's label to every row in this file
                labels = np.full((features.shape[0],), label_value)
                all_labels.append(labels)

    return np.vstack(all_features), np.concatenate(all_labels)

# 3) Set your paths
# Ensure your Drive has: /Training/Drowsy/ and /Training/Alert/
train_base = '/content/drive/MyDrive/GP/CSV/training'
test_base = '/content/drive/MyDrive/GP/CSV/testing'

X_train_raw, y_train = load_labeled_data(train_base)
X_test_raw, y_test = load_labeled_data(test_base)

# 4) Preprocessing & Reshaping
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Reshape to (samples, time_steps, features) ***** time step check??
X_train = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# =========================
# 5) 2clayer
# =========================
model = Sequential()

model.add(LSTM(32, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(16))

model.add(Dense(1, activation='linear'))

# =========================
# 6)
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='mse',
    metrics=['mae']
)

# =========================
# 7)
# =========================
model.fit(X_train, y_train, epochs=20, verbose=1)

# =========================
# 8)
# =========================
pred = model.predict(X_test)

print("Predictions:")
print(pred)

# =========================
# 9)
# =========================


print("Actual vs Predicted (first 20):")
for i in range(20):
    print(f"Actual: {y_test[i]:<5} | Predicted: {pred[i][0]:.4f}")

#classes = [classify(p[0]) for p in pred]

#print("Final Classes:")
#print(classes)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 26.8581 - mae: 2.7457
Epoch 2/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 25.6285 - mae: 2.8322
Epoch 3/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 23.1236 - mae: 3.0305
Epoch 4/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 20.6232 - mae: 3.3258
Epoch 5/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 19.4688 - mae: 3.5931
Epoch 6/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 19.1517 - mae: 3.7480
Epoch 7/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 19.0584 - mae: 3.7840
Epoch 8/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 19.0055 - mae: 3.8104
Epoch 9/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.9621 - mae: 3.8048
Epoch 10/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 18.9234 - mae: 3.8088
Epoch 11/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 18.8867 - mae: 3.8028
Epoch 12/20
425/425 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 18.8541 - mae: 3.7965
Epoch 13/20
4